## Import Dependencies

In [ ]:
import random
import re
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Random State & Device Initialization

In [ ]:
SEED = 42


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

In [ ]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

## Formatting

In [ ]:
PLOT_DPI = 165

COLORS = {
    "train": "#2563EB",
    "valid": "#F97316",
    "test": "#16A34A",
    "adam": "#2563EB",
    "lbfgs": "#9333EA",
    "data": "#0EA5E9",
    "physics": "#DC2626",
    "reflection": "#16A34A",
    "ideal": "#111827",
    "gray": "#6B7280",
    "light_gray": "#E5E7EB",
    "purple": "#7C3AED",
    "amber": "#F59E0B",
}

SPLIT_COLORS = {
    "train": COLORS["train"],
    "valid": COLORS["valid"],
    "test": COLORS["test"],
}

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "figure.dpi": PLOT_DPI,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11.5,
    "legend.fontsize": 9.5,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titleweight": "semibold",
    "axes.labelcolor": "#111827",
    "xtick.color": "#374151",
    "ytick.color": "#374151",
    "axes.edgecolor": "#D1D5DB",
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "grid.linewidth": 0.8,
    "grid.color": "#9CA3AF",
    "lines.linewidth": 2.1,
    "legend.frameon": True,
    "legend.framealpha": 0.94,
    "legend.edgecolor": "#E5E7EB",
})


def polish_axes(
    ax,
    title: str | None = None,
    xlabel: str | None = None,
    ylabel: str | None = None,
    legend: bool = False,
) -> None:
    if title is not None:
        ax.set_title(title, pad=11)
    if xlabel is not None:
        ax.set_xlabel(xlabel, labelpad=7)
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=7)

    ax.grid(True, alpha=0.22, linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_alpha(0.7)
    ax.spines["bottom"].set_alpha(0.7)
    ax.tick_params(axis="both", which="major", length=4, width=0.8, color="#6B7280")

    if legend:
        leg = ax.legend()
        if leg is not None:
            leg.get_frame().set_facecolor("white")
            leg.get_frame().set_edgecolor("#E5E7EB")
            leg.get_frame().set_linewidth(0.9)
            leg.get_frame().set_alpha(0.95)


def smooth_series(values, window: int = 35) -> np.ndarray:
    return pd.Series(values).rolling(window=window, min_periods=1, center=False).mean().to_numpy()


def log10_safe(values, eps: float = 1e-16) -> np.ndarray:
    return np.log10(np.asarray(values).clip(min=eps))


def plot_density_hist(
    ax,
    values,
    *,
    bins: int = 80,
    color: str,
    label: str,
    alpha: float = 0.20,
    linewidth: float = 2.2,
) -> None:
    ax.hist(
        values,
        bins=bins,
        density=True,
        histtype="stepfilled",
        alpha=alpha,
        color=color,
        edgecolor=color,
        linewidth=1.0,
        label=label,
    )
    ax.hist(
        values,
        bins=bins,
        density=True,
        histtype="step",
        color=color,
        linewidth=linewidth,
    )


def plot_ecdf(
    ax,
    values,
    *,
    color: str,
    label: str,
    linewidth: float = 2.2,
) -> None:
    x = np.sort(np.asarray(values))
    y = np.linspace(0.0, 1.0, len(x), endpoint=True)
    ax.plot(x, y, color=color, linewidth=linewidth, label=label)


def compact_count(n: int) -> str:
    n = int(n)
    if n % 1000 == 0:
        return f"{n // 1000}k"
    return str(n)


def compact_float(value: float) -> str:
    value = float(value)
    if value == 0:
        return "0"
    if abs(value - round(value)) < 1e-12:
        return str(int(round(value)))
    text = f"{value:g}"
    return text.replace(".", "p").replace("-", "m").replace("+", "")


def save_current_figure(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")


def save_dataframe_as_png(df_to_save: pd.DataFrame, path: Path, title: str | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    display_df = df_to_save.copy()
    for col in display_df.columns:
        if pd.api.types.is_numeric_dtype(display_df[col]):
            display_df[col] = display_df[col].map(lambda x: f"{x:.6g}" if pd.notna(x) else "")

    fig_height = max(1.8, 0.42 * (len(display_df) + 1) + (0.35 if title else 0.0))
    fig_width = max(8.0, 1.35 * len(display_df.columns))

    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=180)
    ax.axis("off")
    if title is not None:
        ax.set_title(title, fontsize=12.5, fontweight="semibold", pad=10)

    table = ax.table(
        cellText=display_df.values,
        colLabels=display_df.columns,
        cellLoc="center",
        colLoc="center",
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8.5)
    table.scale(1.0, 1.25)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#E5E7EB")
        cell.set_linewidth(0.7)
        if row == 0:
            cell.set_text_props(weight="semibold", color="#111827")
            cell.set_facecolor("#F3F4F6")
        else:
            cell.set_facecolor("white")

    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)


def save_dataframe_text(df_to_save: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(df_to_save.to_string(index=False), encoding="utf-8")

## Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    notebook_dir: Path = Path.cwd()
    project_root: Path = notebook_dir.parent if notebook_dir.name.lower() == "pinn" else notebook_dir

    data_path: Path = project_root / "storage" / "dataset_10k.csv"
    saved_models_dir: Path = (
        notebook_dir / "saved_models"
        if notebook_dir.name.lower() == "pinn"
        else project_root / "PINN" / "saved_models"
    )
    visualization_dir: Path = project_root / "visualization"

    load_existing_model: bool = False
    model_checkpoint_path: Path | None = None
    save_model_after_training: bool = True
    model_run_id: int | None = None

    plate_rho: float = 7800.0
    plate_h: float = 0.002
    plate_E: float = 2.0e11
    plate_nu: float = 0.3

    omega0: float = 2.0 * np.pi * 100.0
    period_a: float = 0.05

    min_R: float = 0.0
    max_R: float = 1.0

    add_noise: bool = False
    noise_level: float = 0.0
    noise_seed: int = 42

    test_size: float = 0.15
    valid_size: float = 0.15
    n_strat_bins: int = 10

    batch_size: int = 256
    use_weighted_sampler: bool = True
    sampler_high_R_scale: float = 1.0
    max_epochs: int = 2000
    learning_rate: float = 5.0e-4
    weight_decay: float = 5.0e-4

    use_lbfgs: bool = True
    lbfgs_lr: float = 1.0
    lbfgs_max_iter: int = 5000
    lbfgs_max_eval: int = 7500
    lbfgs_history_size: int = 100
    lbfgs_tolerance_grad: float = 1.0e-9
    lbfgs_tolerance_change: float = 1.0e-11
    lbfgs_log_every: int = 10
    lbfgs_valid_log_every: int = 25

    lambda_phys: float = 0.5
    lambda_R: float = 2.0
    R_loss_eps: float = 1.0e-6
    R_loss_log_weight: float = 0.85
    R_loss_sqrt_weight: float = 0.15

    patience: int = 180
    min_delta: float = 1.0e-7

    hidden_dim: int = 160
    num_hidden_layers: int = 4
    dropout: float = 0.02

    grad_clip_norm: float = 5.0
    scheduler_factor: float = 0.5
    scheduler_patience: int = 60

    numerical_eps: float = 1.0e-12
    spectrum_n_terms: int = 80
    resonator_damping_ratio: float = 0.01
    resonator_mu_scale: float = 1.0

    fixed_spectrum_points: int = 500
    reference_psi: float = np.pi / 6.0
    near_zero_delta_y: float = 1.0e-4

    geometry_grid_size: int = 72
    geometry_kappa_quantile_low: float = 0.85
    geometry_kappa_quantile_high: float = 0.95
    zeta_error_eps: float = 1.0e-12

    plot_sample_size: int = 10_000
    relative_error_min_R: float = 1.0e-4

    use_spectrum_augmentation: bool = True
    spectrum_points_per_target_case: int = 800
    spectrum_random_cases: int = 6
    spectrum_points_per_random_case: int = 180
    spectrum_augmentation_seed: int = 2026


cfg = Config()

print("Current directory:", Path.cwd())
print("Dataset path:", cfg.data_path)
print("Saved models root:", cfg.saved_models_dir)
print("Visualization root:", cfg.visualization_dir)
print(cfg)

## Spectrum Augmentation